# PCOS ECG-Centric Synthetic Data Generation and Validation

This single-notebook Google Colab pipeline:

1. Audits the uploaded 10-subject PCOS dataset.
2. Trains CTGAN, Gaussian Copula, and a mixed-type Bayesian network.
3. Generates exactly 30 rows from each synthesizer.
4. Runs SDMetrics quality reports and independent statistical validation.
5. Creates publication-quality correlation and PCA figures.
6. Ranks the synthesizers, selects a winner, and exports every result.

> **Scientific caution:** With only 10 source subjects, all estimates have high uncertainty. Synthetic rows are not independent clinical observations and must not be used as a substitute for external validation, prevalence estimation, or efficacy/safety evidence.


## 0. Install dependencies

In [ ]:
# Install/upgrade the current public SDV stack and scientific Python dependencies.
# In Colab, run this cell once. A runtime restart is normally not required.
%pip install -q -U sdv sdmetrics pandas numpy scipy scikit-learn matplotlib seaborn networkx


## 1. Imports, reproducibility, and output directories

In [ ]:
import os
import sys
import json
import math
import random
import shutil
import warnings
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import normalized_mutual_info_score
from sklearn.linear_model import LinearRegression

from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer, GaussianCopulaSynthesizer
from sdmetrics.reports.single_table import QualityReport

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
N_SYNTHETIC = 30
TABLE_NAME = "pcos_ecg"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

OUTPUT_ROOT = Path("outputs")
SYNTH_DIR = OUTPUT_ROOT / "synthetic_datasets"
REPORT_DIR = OUTPUT_ROOT / "reports"
PLOT_DIR = OUTPUT_ROOT / "plots"
COMPARISON_DIR = OUTPUT_ROOT / "comparison"

for directory in [OUTPUT_ROOT, SYNTH_DIR, REPORT_DIR, PLOT_DIR, COMPARISON_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
})

print(f"Python: {sys.version.split()[0]}")
try:
    import sdv
    import sdmetrics
    print(f"SDV: {sdv.__version__}")
    print(f"SDMetrics: {sdmetrics.__version__}")
except Exception:
    pass
print(f"Random seed: {RANDOM_STATE}")
print(f"Output root: {OUTPUT_ROOT.resolve()}")


## 2. Dataset schema and automatic loading

In [ ]:
IDENTIFIER = "Subject_ID"
LABEL_COLUMNS = ["Diagnosis", "Target_Class"]
CATEGORICAL_COLUMNS = [
    "Diagnosis",
    "Target_Class",
    "Cycle Regularity",
    "Family History (PCOS/Diabetes)",
]
NUMERICAL_FEATURES = [
    "HRV_SDNN_Estimate (ms)",
    "Mean_RR_Interval (s)",
    "QTc_Bazett (s)",
    "QT_Interval (s)",
    "P_Wave_Dispersion (s)",
    "PR_Interval (s)",
    "QRS_Duration (s)",
    "T_Wave_Duration (s)",
    "SDNN_MeanRR_Ratio",
    "QTc_MeanRR_Ratio",
    "PWaveDisp_x_QTc",
    "HRVSDNN_x_QTc",
    "Cycle Length",
    "BMI",
]
EXPECTED_COLUMNS = [IDENTIFIER] + LABEL_COLUMNS + [
    c for c in NUMERICAL_FEATURES + CATEGORICAL_COLUMNS if c not in LABEL_COLUMNS
]


def normalize_column_names(frame):
    result = frame.copy()
    result.columns = [
        str(c).strip().replace("\n", " ").replace("\r", " ") for c in result.columns
    ]
    return result


def locate_input_csv():
    preferred = [
        Path("/content/PCOS_ECG_Centric_16_Features_Corrected.csv"),
        Path("PCOS_ECG_Centric_16_Features_Corrected.csv"),
    ]
    for path in preferred:
        if path.exists():
            return path

    candidates = list(Path("/content").glob("*.csv")) if Path("/content").exists() else []
    candidates += list(Path(".").glob("*.csv"))

    scored = []
    for path in dict.fromkeys(candidates):
        try:
            header = normalize_column_names(pd.read_csv(path, nrows=2))
            score = len(set(EXPECTED_COLUMNS).intersection(header.columns))
            scored.append((score, path))
        except Exception:
            continue

    if scored:
        score, path = max(scored, key=lambda item: item[0])
        if score >= max(4, len(EXPECTED_COLUMNS) // 2):
            return path

    # Colab fallback: open an upload dialog only when no suitable CSV is present.
    try:
        from google.colab import files
        print("No matching CSV found. Select the PCOS CSV in the upload dialog.")
        uploaded = files.upload()
        csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csv_names:
            raise FileNotFoundError("No CSV file was uploaded.")
        return Path(csv_names[0])
    except ImportError as exc:
        raise FileNotFoundError(
            "Could not find PCOS_ECG_Centric_16_Features_Corrected.csv."
        ) from exc


INPUT_PATH = locate_input_csv()
raw_data = normalize_column_names(pd.read_csv(INPUT_PATH))

missing_required = [c for c in EXPECTED_COLUMNS if c not in raw_data.columns]
if missing_required:
    raise ValueError(
        "The input CSV is missing required columns:\n- "
        + "\n- ".join(missing_required)
        + f"\n\nColumns found: {raw_data.columns.tolist()}"
    )

# Keep the requested columns in a stable order and ignore accidental export columns.
data = raw_data[EXPECTED_COLUMNS].copy()

# Coerce declared types while preserving an audit of the untouched input above.
for column in NUMERICAL_FEATURES:
    data[column] = pd.to_numeric(data[column], errors="coerce")
for column in CATEGORICAL_COLUMNS:
    data[column] = data[column].astype("string")
data[IDENTIFIER] = data[IDENTIFIER].astype("string")

# Models require complete values. Use conservative, explicit imputation only if needed.
for column in NUMERICAL_FEATURES:
    if data[column].isna().any():
        data[column] = data[column].fillna(data[column].median())
for column in CATEGORICAL_COLUMNS:
    if data[column].isna().any():
        mode = data[column].mode(dropna=True)
        fill = mode.iloc[0] if len(mode) else "Missing"
        data[column] = data[column].fillna(fill)
if data[IDENTIFIER].isna().any() or data[IDENTIFIER].duplicated().any():
    data[IDENTIFIER] = [f"SUBJ_{i:04d}" for i in range(1, len(data) + 1)]

print(f"Loaded: {INPUT_PATH}")
print(f"Validated modeling shape: {data.shape}")
display(data.head())


## 3. Data audit

In [ ]:
print("=" * 80)
print("DATASET SHAPE")
print("=" * 80)
print(raw_data.shape)

print("\n" + "=" * 80)
print("DATA TYPES")
print("=" * 80)
display(raw_data.dtypes.rename("dtype").to_frame())

print("\n" + "=" * 80)
print("MISSING VALUE REPORT")
print("=" * 80)
missing_report = pd.DataFrame({
    "Missing_Count": raw_data.isna().sum(),
    "Missing_Percent": raw_data.isna().mean().mul(100).round(2),
})
display(missing_report)
missing_report.to_csv(REPORT_DIR / "missing_value_report.csv")

print("\n" + "=" * 80)
print("CLASS DISTRIBUTIONS")
print("=" * 80)
class_distribution_rows = []
for column in LABEL_COLUMNS:
    distribution = raw_data[column].value_counts(dropna=False)
    percentages = raw_data[column].value_counts(dropna=False, normalize=True).mul(100)
    table = pd.DataFrame({"Count": distribution, "Percent": percentages.round(2)})
    print(f"\n{column}")
    display(table)
    for value, row in table.iterrows():
        class_distribution_rows.append({
            "Column": column, "Class": value,
            "Count": row["Count"], "Percent": row["Percent"],
        })
pd.DataFrame(class_distribution_rows).to_csv(
    REPORT_DIR / "class_distributions.csv", index=False
)

print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
summary_statistics = data.describe(include="all").transpose()
display(summary_statistics)
summary_statistics.to_csv(REPORT_DIR / "summary_statistics.csv")


## 4. SDV metadata

`Subject_ID` is treated as a primary-key identifier. The four declared categorical columns are explicitly typed; all predictive measurements are numerical. The metadata is saved with the results for reproducibility.


In [ ]:
metadata = Metadata.detect_from_dataframe(
    data=data,
    table_name=TABLE_NAME,
    infer_sdtypes=True,
    infer_keys=None,
)

metadata.update_column(column_name=IDENTIFIER, sdtype="id")
metadata.set_primary_key(column_name=IDENTIFIER)
for column in CATEGORICAL_COLUMNS:
    metadata.update_column(column_name=column, sdtype="categorical")
for column in NUMERICAL_FEATURES:
    metadata.update_column(column_name=column, sdtype="numerical")

metadata.validate()
metadata.validate_table(data)

metadata_path = REPORT_DIR / "metadata.json"
if metadata_path.exists():
    metadata_path.unlink()
metadata.save_to_json(filepath=str(metadata_path))

metadata_dict = metadata.to_dict()
display(pd.DataFrame.from_dict(
    metadata_dict["tables"][TABLE_NAME]["columns"], orient="index"
))


## 5. Bayesian network synthesizer

SDV Community currently exposes CTGAN and Gaussian Copula for this task, but not a public class named `BayesianNetworkSynthesizer`. The implementation below is a mixed-type, tree-structured Bayesian network:

- A maximum-spanning dependency tree is learned using normalized mutual information.
- Categorical conditional probability tables use Laplace smoothing.
- Numerical children use conditional empirical distributions or linear-Gaussian conditionals.
- The fixed seed makes both structure learning and sampling reproducible.

This is intentionally compact and robust for a dataset of only 10 subjects.


In [ ]:
class BayesianNetworkSynthesizer:
    """Mixed-type tree Bayesian network for very small tabular datasets."""

    def __init__(self, categorical_columns, numerical_columns, id_column, random_state=42):
        self.categorical_columns = list(categorical_columns)
        self.numerical_columns = list(numerical_columns)
        self.id_column = id_column
        self.random_state = random_state
        self.rng = np.random.default_rng(random_state)

    @staticmethod
    def _discretize(series, max_bins=4):
        if not pd.api.types.is_numeric_dtype(series):
            return series.astype(str).fillna("Missing")
        unique = series.nunique(dropna=True)
        if unique <= 1:
            return pd.Series(["constant"] * len(series), index=series.index)
        bins = min(max_bins, unique)
        try:
            return pd.qcut(series, q=bins, duplicates="drop").astype(str)
        except Exception:
            return pd.cut(series, bins=bins, duplicates="drop").astype(str)

    def fit(self, frame):
        self.data = frame.copy().reset_index(drop=True)
        self.model_columns = [c for c in frame.columns if c != self.id_column]
        discretized = {
            column: self._discretize(self.data[column])
            for column in self.model_columns
        }

        graph = nx.Graph()
        graph.add_nodes_from(self.model_columns)
        for i, left in enumerate(self.model_columns):
            for right in self.model_columns[i + 1:]:
                weight = normalized_mutual_info_score(discretized[left], discretized[right])
                graph.add_edge(left, right, weight=float(weight) + 1e-12)

        tree = nx.maximum_spanning_tree(graph, weight="weight")
        preferred_roots = ["Target_Class", "Diagnosis"]
        self.root = next((c for c in preferred_roots if c in tree), self.model_columns[0])

        self.parent = {self.root: None}
        queue = deque([self.root])
        while queue:
            node = queue.popleft()
            for neighbor in tree.neighbors(node):
                if neighbor not in self.parent:
                    self.parent[neighbor] = node
                    queue.append(neighbor)

        self.order = list(nx.bfs_tree(tree, self.root).nodes())
        self.tree = tree
        return self

    def _sample_categorical(self, column, parent, generated, row_index):
        values = self.data[column].astype(str)
        categories = sorted(values.unique().tolist())
        subset = values
        if parent is not None:
            parent_value = generated[parent][row_index]
            parent_real = self.data[parent]
            if parent in self.categorical_columns:
                mask = parent_real.astype(str) == str(parent_value)
            else:
                nearest = np.abs(parent_real.astype(float) - float(parent_value)).argsort()[:3]
                mask = self.data.index.isin(nearest)
            if np.sum(mask) >= 1:
                subset = values[mask]
        counts = subset.value_counts().reindex(categories, fill_value=0).astype(float) + 0.5
        probabilities = (counts / counts.sum()).to_numpy()
        return self.rng.choice(categories, p=probabilities)

    def _sample_numerical(self, column, parent, generated, row_index):
        y = self.data[column].astype(float).to_numpy()
        if parent is None:
            sampled = self.rng.choice(y)
            noise = self.rng.normal(0, np.std(y, ddof=1) * 0.05 if len(y) > 1 else 0)
            return sampled + noise

        parent_value = generated[parent][row_index]
        if parent in self.categorical_columns:
            mask = self.data[parent].astype(str) == str(parent_value)
            candidates = y[mask.to_numpy()]
            if len(candidates) == 0:
                candidates = y
            sampled = self.rng.choice(candidates)
            scale = np.std(candidates, ddof=1) if len(candidates) > 1 else np.std(y, ddof=1)
            return sampled + self.rng.normal(0, (scale if np.isfinite(scale) else 0) * 0.05)

        x = self.data[[parent]].astype(float).to_numpy()
        model = LinearRegression().fit(x, y)
        fitted = model.predict(x)
        residuals = y - fitted
        prediction = model.predict(np.array([[float(parent_value)]]))[0]
        return prediction + self.rng.choice(residuals)

    def sample(self, num_rows):
        generated = {column: [] for column in self.model_columns}
        for row_index in range(num_rows):
            for column in self.order:
                parent = self.parent[column]
                if column in self.categorical_columns:
                    value = self._sample_categorical(column, parent, generated, row_index)
                else:
                    value = self._sample_numerical(column, parent, generated, row_index)
                    real = self.data[column].astype(float)
                    value = float(np.clip(value, real.min(), real.max()))
                generated[column].append(value)

        result = pd.DataFrame(generated)
        result.insert(
            0, self.id_column,
            [f"BN_SYN_{i:04d}" for i in range(1, num_rows + 1)]
        )
        return result[self.data.columns]

    def structure_table(self):
        return pd.DataFrame([
            {"Child": child, "Parent": parent}
            for child, parent in self.parent.items()
        ])


## 6. Train all three synthesizers

In [ ]:
def reset_all_seeds(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def set_sdv_seed(synthesizer, seed=RANDOM_STATE):
    # Supported by current SDV base synthesizers; fallback keeps NumPy/Torch seeded.
    if hasattr(synthesizer, "set_random_state"):
        synthesizer.set_random_state(seed)


trained_models = {}
training_errors = {}

print("Training CTGAN...")
try:
    reset_all_seeds()
    ctgan = CTGANSynthesizer(
        metadata,
        epochs=500,
        batch_size=100,
        pac=10,
        enforce_min_max_values=True,
        enforce_rounding=True,
        verbose=False,
        enable_gpu=True,
    )
    set_sdv_seed(ctgan)
    ctgan.fit(data)
    trained_models["CTGAN"] = ctgan
    print("CTGAN trained successfully.")
except Exception as exc:
    training_errors["CTGAN"] = repr(exc)
    print(f"CTGAN failed: {exc}")

print("\nTraining Gaussian Copula...")
try:
    reset_all_seeds()
    gaussian = GaussianCopulaSynthesizer(
        metadata,
        enforce_min_max_values=True,
        enforce_rounding=True,
        default_distribution="beta",
    )
    set_sdv_seed(gaussian)
    gaussian.fit(data)
    trained_models["GaussianCopula"] = gaussian
    print("Gaussian Copula trained successfully.")
except Exception as exc:
    training_errors["GaussianCopula"] = repr(exc)
    print(f"Gaussian Copula failed: {exc}")

print("\nTraining Bayesian Network...")
try:
    bayesian = BayesianNetworkSynthesizer(
        categorical_columns=CATEGORICAL_COLUMNS,
        numerical_columns=NUMERICAL_FEATURES,
        id_column=IDENTIFIER,
        random_state=RANDOM_STATE,
    ).fit(data)
    trained_models["BayesianNetwork"] = bayesian
    bayesian.structure_table().to_csv(
        REPORT_DIR / "bayesian_network_structure.csv", index=False
    )
    print("Bayesian Network trained successfully.")
    display(bayesian.structure_table())
except Exception as exc:
    training_errors["BayesianNetwork"] = repr(exc)
    print(f"Bayesian Network failed: {exc}")

if training_errors:
    with open(REPORT_DIR / "training_errors.json", "w") as handle:
        json.dump(training_errors, handle, indent=2)

required_models = {"CTGAN", "GaussianCopula", "BayesianNetwork"}
missing_models = required_models.difference(trained_models)
if missing_models:
    raise RuntimeError(
        f"Required synthesizer training failed: {sorted(missing_models)}. "
        f"See {REPORT_DIR / 'training_errors.json'}."
    )


## 7. Generate and save exactly 30 rows per synthesizer

In [ ]:
OUTPUT_FILENAMES = {
    "CTGAN": "ctgan_30.csv",
    "GaussianCopula": "gaussiancopula_30.csv",
    "BayesianNetwork": "bayesiannetwork_30.csv",
}


def harmonize_output(frame, model_name):
    result = frame.copy()[data.columns]
    for column in NUMERICAL_FEATURES:
        result[column] = pd.to_numeric(result[column], errors="coerce")
        result[column] = result[column].fillna(data[column].median())
        result[column] = result[column].clip(data[column].min(), data[column].max())
    for column in CATEGORICAL_COLUMNS:
        result[column] = result[column].astype(str)
        valid = set(data[column].astype(str))
        fallback = data[column].astype(str).mode().iloc[0]
        result.loc[~result[column].isin(valid), column] = fallback
    result[IDENTIFIER] = [
        f"{model_name.upper()}_{i:04d}" for i in range(1, len(result) + 1)
    ]
    return result


synthetic_datasets = {}
for model_name, model in trained_models.items():
    reset_all_seeds()
    sampled = model.sample(num_rows=N_SYNTHETIC)
    sampled = harmonize_output(sampled, model_name)
    if sampled.shape[0] != N_SYNTHETIC:
        raise RuntimeError(f"{model_name} generated {sampled.shape[0]} rows, not 30.")
    synthetic_datasets[model_name] = sampled
    output_path = SYNTH_DIR / OUTPUT_FILENAMES[model_name]
    sampled.to_csv(output_path, index=False)
    print(f"Saved {model_name}: {output_path} | shape={sampled.shape}")

display(synthetic_datasets["CTGAN"].head())


## 8. SDMetrics quality reports

In [ ]:
def single_table_metadata_for_metrics(metadata_object):
    as_dict = metadata_object.to_dict()
    # QualityReport expects the table-level metadata dictionary.
    if "tables" in as_dict:
        return as_dict["tables"][TABLE_NAME]
    return as_dict


metrics_metadata = single_table_metadata_for_metrics(metadata)
quality_rows = []
quality_reports = {}

for model_name, synthetic in synthetic_datasets.items():
    print(f"\nGenerating SDMetrics Quality Report: {model_name}")
    report = QualityReport()
    report.generate(
        real_data=data,
        synthetic_data=synthetic,
        metadata=metrics_metadata,
        verbose=False,
    )
    quality_reports[model_name] = report
    properties = report.get_properties()
    property_scores = dict(zip(properties["Property"], properties["Score"]))
    row = {
        "Synthesizer": model_name,
        "Overall Quality Score": float(report.get_score()),
        "Column Shapes Score": float(property_scores.get("Column Shapes", np.nan)),
        "Column Pair Trends Score": float(
            property_scores.get("Column Pair Trends", np.nan)
        ),
    }
    quality_rows.append(row)

    properties.to_csv(
        REPORT_DIR / f"{model_name.lower()}_quality_properties.csv", index=False
    )
    for property_name in ["Column Shapes", "Column Pair Trends"]:
        try:
            details = report.get_details(property_name=property_name)
            safe_name = property_name.lower().replace(" ", "_")
            details.to_csv(
                REPORT_DIR / f"{model_name.lower()}_{safe_name}_details.csv",
                index=False,
            )
        except Exception as exc:
            print(f"Could not save {property_name} details for {model_name}: {exc}")

quality_scores = pd.DataFrame(quality_rows)
quality_scores.to_csv(REPORT_DIR / "quality_scores.csv", index=False)
display(quality_scores.style.format({
    "Overall Quality Score": "{:.4f}",
    "Column Shapes Score": "{:.4f}",
    "Column Pair Trends Score": "{:.4f}",
}))


## 9. Correlation preservation: matrices, differences, and heatmaps

In [ ]:
def save_heatmap(matrix, title, path, center=0, cmap="coolwarm"):
    size = max(10, 0.65 * len(matrix.columns))
    fig, ax = plt.subplots(figsize=(size, size * 0.85))
    sns.heatmap(
        matrix,
        cmap=cmap,
        center=center,
        square=True,
        linewidths=0.35,
        cbar_kws={"shrink": 0.8},
        ax=ax,
    )
    ax.set_title(title, pad=14)
    plt.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


original_corr = data[NUMERICAL_FEATURES].corr(method="pearson")
original_corr.to_csv(REPORT_DIR / "original_pearson_correlation.csv")
save_heatmap(
    original_corr,
    "Original Pearson Correlation Matrix",
    PLOT_DIR / "original_pearson_correlation.png",
)

correlation_error_rows = []
for model_name, synthetic in synthetic_datasets.items():
    synthetic_corr = synthetic[NUMERICAL_FEATURES].corr(method="pearson")
    difference = (original_corr - synthetic_corr).abs()
    slug = model_name.lower()

    synthetic_corr.to_csv(REPORT_DIR / f"{slug}_pearson_correlation.csv")
    difference.to_csv(REPORT_DIR / f"{slug}_correlation_difference.csv")

    save_heatmap(
        synthetic_corr,
        f"{model_name}: Synthetic Pearson Correlation",
        PLOT_DIR / f"{slug}_pearson_correlation.png",
    )
    save_heatmap(
        difference,
        f"{model_name}: Absolute Correlation Difference",
        PLOT_DIR / f"{slug}_correlation_difference.png",
        center=None,
        cmap="magma",
    )

    upper_triangle = difference.to_numpy()[np.triu_indices_from(difference, k=1)]
    correlation_error_rows.append({
        "Synthesizer": model_name,
        "Mean Correlation Difference": float(np.nanmean(upper_triangle)),
    })

correlation_errors = pd.DataFrame(correlation_error_rows)
correlation_errors.to_csv(REPORT_DIR / "correlation_error_summary.csv", index=False)
display(correlation_errors)


## 10. Distribution preservation: KS and Wasserstein metrics

In [ ]:
distribution_tables = {}
distribution_summary_rows = []

for model_name, synthetic in synthetic_datasets.items():
    rows = []
    for column in NUMERICAL_FEATURES:
        real_values = data[column].dropna().astype(float).to_numpy()
        synthetic_values = synthetic[column].dropna().astype(float).to_numpy()
        ks_result = ks_2samp(real_values, synthetic_values, alternative="two-sided")
        rows.append({
            "Feature": column,
            "KS Statistic": float(ks_result.statistic),
            "KS p-value": float(ks_result.pvalue),
            "Wasserstein Distance": float(
                wasserstein_distance(real_values, synthetic_values)
            ),
        })

    table = pd.DataFrame(rows).sort_values(
        ["KS Statistic", "Wasserstein Distance"], ascending=[True, True]
    )
    table.insert(0, "Rank", np.arange(1, len(table) + 1))
    distribution_tables[model_name] = table
    table.to_csv(
        REPORT_DIR / f"{model_name.lower()}_distribution_metrics.csv", index=False
    )
    distribution_summary_rows.append({
        "Synthesizer": model_name,
        "Mean KS Statistic": table["KS Statistic"].mean(),
        "Mean Wasserstein Distance": table["Wasserstein Distance"].mean(),
    })
    print(f"\n{model_name} numerical feature ranking")
    display(table.style.format({
        "KS Statistic": "{:.4f}",
        "KS p-value": "{:.4g}",
        "Wasserstein Distance": "{:.6f}",
    }))

distribution_summary = pd.DataFrame(distribution_summary_rows)
distribution_summary.to_csv(
    REPORT_DIR / "distribution_error_summary.csv", index=False
)


## 11. PCA comparisons

PCA is fitted separately for each original-versus-synthetic comparison after scaling the combined numerical data. This puts both groups in the same two-dimensional coordinate system for each figure.


In [ ]:
def plot_pca_comparison(real, synthetic, model_name, output_path):
    combined = pd.concat(
        [
            real[NUMERICAL_FEATURES].assign(Source="Original"),
            synthetic[NUMERICAL_FEATURES].assign(Source=model_name),
        ],
        ignore_index=True,
    )
    x = combined[NUMERICAL_FEATURES].astype(float)
    scaled = StandardScaler().fit_transform(x)
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    coordinates = pca.fit_transform(scaled)

    plot_data = pd.DataFrame({
        "PC1": coordinates[:, 0],
        "PC2": coordinates[:, 1],
        "Source": combined["Source"].to_numpy(),
    })

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(
        data=plot_data,
        x="PC1",
        y="PC2",
        hue="Source",
        style="Source",
        s=90,
        alpha=0.82,
        palette=["#1f77b4", "#d62728"],
        ax=ax,
    )
    variance = pca.explained_variance_ratio_ * 100
    ax.set_xlabel(f"PC1 ({variance[0]:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({variance[1]:.1f}% variance)")
    ax.set_title(f"PCA Comparison: Original vs {model_name}")
    ax.legend(title="")
    plt.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


for model_name, synthetic in synthetic_datasets.items():
    plot_pca_comparison(
        data,
        synthetic,
        model_name,
        PLOT_DIR / f"pca_original_vs_{model_name.lower()}.png",
    )


## 12. Final synthesizer comparison and automatic winner selection

In [ ]:
comparison = (
    quality_scores
    .merge(correlation_errors, on="Synthesizer", how="inner")
    .merge(distribution_summary, on="Synthesizer", how="inner")
)

# Lexicographic priority exactly follows the requested decision policy:
# 1) highest quality, 2) lowest correlation error,
# 3) lowest KS error, then lowest Wasserstein distance.
comparison = comparison.sort_values(
    by=[
        "Overall Quality Score",
        "Mean Correlation Difference",
        "Mean KS Statistic",
        "Mean Wasserstein Distance",
    ],
    ascending=[False, True, True, True],
).reset_index(drop=True)
comparison.insert(0, "Rank", np.arange(1, len(comparison) + 1))

comparison_path = COMPARISON_DIR / "synthesizer_comparison.csv"
comparison.to_csv(comparison_path, index=False)

best_synthesizer = comparison.loc[0, "Synthesizer"]
best_quality_score = float(comparison.loc[0, "Overall Quality Score"])
(COMPARISON_DIR / "best_synthesizer.txt").write_text(
    best_synthesizer + "\n", encoding="utf-8"
)

display(comparison.style.format({
    "Overall Quality Score": "{:.4f}",
    "Column Shapes Score": "{:.4f}",
    "Column Pair Trends Score": "{:.4f}",
    "Mean Correlation Difference": "{:.4f}",
    "Mean KS Statistic": "{:.4f}",
    "Mean Wasserstein Distance": "{:.6f}",
}))
print(f"Best synthesizer: {best_synthesizer}")


## 13. Export archive and final notebook summary

In [ ]:
# Create a single downloadable archive while retaining the full directory tree.
archive_path = shutil.make_archive(
    base_name="PCOS_synthetic_data_results",
    format="zip",
    root_dir=OUTPUT_ROOT,
)

saved_files = sorted(
    str(path) for path in OUTPUT_ROOT.rglob("*") if path.is_file()
)

print("\n" + "=" * 88)
print("FINAL NOTEBOOK SUMMARY")
print("=" * 88)
print(f"Original Samples: {len(data)}")
print("\nGenerated Datasets:")
for filename in OUTPUT_FILENAMES.values():
    print(f"* {filename}")
print(f"\nBest Synthesizer:\n{best_synthesizer}")
print(f"\nQuality Score:\n{best_quality_score:.6f}")
print("\nFiles Saved:")
for filename in saved_files:
    print(f"* {filename}")
print(f"* {archive_path}")
print("\nScientific note: synthetic data quality from n=10 is exploratory, not clinical validation.")

# Automatically trigger a browser download when running in Google Colab.
try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print(f"Archive available at: {Path(archive_path).resolve()}")
except Exception as exc:
    print(f"Automatic download was unavailable: {exc}")
    print(f"Archive remains saved at: {Path(archive_path).resolve()}")
